# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [8]:
context = [ {'role':'system', 'content':"""
 CREATE TABLE employees (
    ID INT PRIMARY KEY,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    birth_date DATE,
    hire_date DATE,
    department VARCHAR(50)
);

CREATE TABLE salary (
    ID INT PRIMARY KEY,
    employee_id INT,
    amount DECIMAL(10,2),
    payment_date DATE,
    FOREIGN KEY (employee_id) REFERENCES employees(ID)
);

CREATE TABLE studies (
    ID INT PRIMARY KEY,
    employee_id INT,
    degree VARCHAR(100),
    institution VARCHAR(100),
    graduation_date DATE,
    FOREIGN KEY (employee_id) REFERENCES employees(ID)
);
"""} ]



In [9]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
SELECT e.first_name, e.last_name, e.hire_date, s.amount
FROM employees e
JOIN salary s ON e.ID = s.employee_id
WHERE e.hire_date > '2000-01-01' AND s.amount > 5000
ORDER BY s.amount DESC;

SELECT e.first_name, e.last_name, st.degree, AVG(s.amount) as avg_salary
FROM employees e
LEFT JOIN studies st ON e.ID = st.employee_id
LEFT JOIN salary s ON e.ID = s.employee_id
GROUP BY e.ID, e.first_name, e.last_name, st.degree
ORDER BY avg_salary DESC;

"""
})

In [10]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [11]:
#new
context_user = context.copy()
print(return_CCRMSQL("""Who has the highest salary?""", context_user))

To find out who has the highest salary, you can run the first query provided in the SQL snippet. This query selects the first name, last name, hire date, and salary amount of employees who were hired after January 1, 2000, and have a salary amount greater than $5000. The results are ordered by the salary amount in descending order, so the employee with the highest salary will be listed first.


In [12]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Who has the highest salary?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of the employee with the highest salary by joining the "employees" table with the "salary" table on the ID_usr column. It then orders the results by salary in descending order and limits the output to only one row, which represents the employee with the highest salary.


In [15]:
#new
print(return_CCRMSQL("Who has the lowest salary?", context_user))

To find the employee with the lowest salary, you can use the following query:

```sql
SELECT e.first_name, e.last_name, s.amount
FROM employees e
JOIN salary s ON e.ID = s.employee_id
ORDER BY s.amount
LIMIT 1;
```

This query will return the employee with the lowest salary, including their first name, last name, and the amount of their salary.


In [16]:
#old
print(return_CCRMSQL("Who has the lowest salary?", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary
LIMIT 1;
```

This SQL query selects the name and salary of the employee with the lowest salary by joining the "employees" and "salary" tables on the employee ID. It then orders the results by salary in ascending order and limits the output to the first row, which represents the employee with the lowest salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [17]:
print(return_CCRMSQL("Who has the lowest salary grouped by studies?", old_context_user))

This is your SQL:
```sql
SELECT e.name, MIN(s.salary) AS lowest_salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
GROUP BY e.name;
```

This SQL query selects the name of employees and their lowest salary by joining the "employees" and "salary" tables on the employee ID. It then groups the results by employee name and calculates the minimum salary for each employee.
